# Feature Track 4b: LLM Agents

The manual request-execute-respond loop from 4a is repetitive. A `ToolAgent` wraps that loop: it runs the LLM, dispatches tool calls, feeds results back, and repeats until the LLM produces a final answer (or hits `max_steps`).

This notebook builds two agents:
- **Simple agent**: minimal setup, two tools, no structured output
- **Complex agent**: custom temperature, structured JSON output schema, same tools

---

## Setup

**Prerequisites:** `conversational_toolkit` and `backend` installed in editable mode. Set `OPENAI_API_KEY`.

In [ ]:
import json

from openai.types.shared_params import ResponseFormatJSONSchema

from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.agents.tool_agent import ToolAgent
from conversational_toolkit.llms.openai import OpenAILLM

from sme_kt_zh_collaboration_rag.feature4_tool_agents import SumTwoNumbers, HalfBoldText

---

## Simple Agent

Two tools are defined: a calculator and a text formatter. The `ToolAgent` runs the tool loop automatically. The agent decides which tool(s) to call, in what order, or whether to answer directly without any tool.

`max_steps` caps the number of LLM-tool iterations to prevent infinite loops.

### Define the Tools

In [ ]:
tool_sum = SumTwoNumbers(
    name="sum_two_numbers",
    description="A tool to sum two numbers. It takes two numbers as input and returns their sum.",
    parameters={
        "type": "object",
        "properties": {
            "number_1": {
                "type": "number",
                "description": "The first number to be summed.",
            },
            "number_2": {
                "type": "number",
                "description": "The second number to be summed.",
            },
        },
        "required": ["number_1", "number_2"],
        "additionalProperties": False,
    },
)

print(await tool_sum.call({"number_1": 5, "number_2": 10}))

In [ ]:
tool_half_bold = HalfBoldText(
    name="half_bold_text",
    description="A tool to bold every other word in a given text. It takes a string as input and returns the modified string with every other word bolded.",
    parameters={
        "type": "object",
        "properties": {
            "text": {
                "type": "string",
                "description": "The input text to be modified.",
            },
        },
        "required": ["text"],
        "additionalProperties": False,
    },
)

print(
    await tool_half_bold.call(
        {"text": "This is an example of the half bold text tool in action."}
    )
)

### Define the Agent

We will make use of the `ToolAgent` class from the toolkit.

Bonus: Below you can find it's implementation, but it's not required to understand deeply.

```python
class ToolAgent(Agent):
    async def answer_stream(self, query_with_context: QueryWithContext) -> AsyncGenerator[AgentAnswer, None]:
        steps = []
        sources: list[Chunk] = []
        messages = [
            LLMMessage(role=Roles.SYSTEM, content=self.system_prompt),
            *query_with_context.history,
            LLMMessage(role=Roles.USER, content=query_with_context.query),
        ]

        while True:
            tool_calls: list[ToolCall] = []
            content = ""
            response_stream = self.llm.generate_stream(messages)
            async for response_chunk in response_stream:
                if response_chunk.content:
                    content += response_chunk.content
                    answer = await self._answer_post_processing(
                        AgentAnswer(content=content, role=Roles.ASSISTANT, sources=sources.copy())
                    )
                    if answer:
                        yield answer
                if response_chunk.tool_calls:
                    tool_calls += response_chunk.tool_calls

            steps.append(
                {
                    "content": content,
                    "tool_calls": tool_calls,
                    "role": Roles.ASSISTANT,
                    "function_name": "llm",
                }
            )
            messages.append(
                LLMMessage(
                    role=Roles.ASSISTANT,
                    content=content,
                    tool_calls=tool_calls,
                )
            )

            if not tool_calls:
                break

            available_functions = {tool.name: tool.call for tool in self.llm.tools} if self.llm.tools else {}

            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = available_functions[function_name]
                function_args = {
                    "_query": query_with_context.query,
                    "_history": query_with_context.history,
                    **json.loads(tool_call.function.arguments),
                }
                function_response = await function_to_call(function_args)
                if "_sources" in function_response:
                    sources += [
                        Chunk(
                            title=chunk.get("title") or "",
                            content=chunk.get("content") or "",
                            mime_type=chunk.get("mime_type") or "",
                            metadata=chunk.get("metadata") or {},
                        )
                        for chunk in function_response.get("_sources", [])
                    ]
                messages.append(self.build_tool_answer(tool_call.id, function_name, function_response))
                steps.append({**function_response, "role": "tool", "function_name": function_name})

            if len(steps) > self.max_steps:
                yield AgentAnswer(content="Request is too complex to execute", role=Roles.ASSISTANT)
                break

        logger.debug(steps)
```

In [ ]:
# Create a custom ToolAgent
class MyAgent(ToolAgent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)


# Define the LLM that will be it's core, and provide it with tools
llm = OpenAILLM(tool_choice="auto", tools=[tool_sum, tool_half_bold])

# Define the prompt
prompt = "You are a helpful assistant, answer shortly."

# Instantiate the agent
simple_agent = MyAgent(
    system_prompt=prompt,
    llm=llm,
    max_steps=5,  # Maximum number of steps the agent can take (avoid infinite loops)
)

### Test the Agent

Now it can select which tool, if any, to use, and iterate in calls.

Our implementation of the agent no longer take `LLMMessage` as input, but `QueryWithContext`, which is basically the user query and the list of previous messages (it simplifies code further on).

```python
class QueryWithContext(BaseModel):
    query: str
    history: list[LLMMessage]
```

#### No Tool Needed

In [ ]:
# Similar to messages, but with history in case it's relevant
query = QueryWithContext(query="How are you?", history=[])

answer = await simple_agent.answer(query)

print(f"\nResponse: \n{answer.content[0].text}")

#### Calculator needed

Look into the logs, the calculator is called.

In [ ]:
query = QueryWithContext(query="Sum 2+59", history=[])

answer = await simple_agent.answer(query)

print(f"\nResponse: \n{answer.content[0].text}")

#### Text Tool needed

Look into the logs, the text formatter is called.

In [ ]:
query = QueryWithContext(
    query="Rewrite the following text with every other word bolded: 'This is an example of the half bold text tool in action.'",
    history=[],
)

answer = await simple_agent.answer(query)

print(f"\nResponse: \n{answer.content[0].text}")

#### Both Tools

Look into the logs, first the calculator is called, and then the text formatter.

In [ ]:
query = QueryWithContext(
    query="First sum 10 and 20, then rewrite the result with every other word bolded by writing a short story about the result (1 sentence).",
    history=[],
)

answer = await simple_agent.answer(query)

print(f"\nResponse: \n{answer.content[0].text}")

---

## Complex Agent

Same two tools, but the LLM is configured with a **structured output schema** (`response_format`) that forces the answer into a JSON object with `answer` and `explanations` fields. This guarantees a parseable, consistent response format.

### Define it

In [ ]:
model = "gpt-4o-mini"
temperature = 0.7

complex_prompt = """You are a helpful assistant, you are responsible for answering questions with clear, professional, and well-supported responses. You must answer like you were a pirate. Justify your tools usage, but use them whenever relevant. Keep it short overall."""

output_schema = {
    "type": "object",
    "description": "The structured output for the answer.",
    "properties": {
        "answer": {
            "type": "string",
            "description": "The answer to the user's question in markdown format.",
        },
        "explanations": {
            "type": "array",
            "description": "List of justifications for the answer (max 3).",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": 3,
        },
    },
    "required": ["answer", "explanations"],
    "additionalProperties": False,
}

response_format: ResponseFormatJSONSchema = {
    "type": "json_schema",
    "json_schema": {
        "name": "AnswerSchema",
        "schema": output_schema,
        "strict": True,
    },
}

tool_choice = "auto"
tools = [tool_sum, tool_half_bold]

llm = OpenAILLM(
    model_name=model,
    temperature=temperature,
    response_format=response_format,
    tool_choice=tool_choice,
    tools=tools,
)

max_nb_steps = 5

complex_agent = MyAgent(
    system_prompt=complex_prompt,
    llm=llm,
    max_steps=max_nb_steps,
)

### No Tool Needed

In [ ]:
query = QueryWithContext(query="How are you?", history=[])

answer = await complex_agent.answer(query)
assert answer.content[0].text is not None
answer_as_json = json.loads(answer.content[0].text)

print("\nAnswer: ", answer_as_json["answer"])
print("\nExplanations: ")
for i, explanation in enumerate(answer_as_json["explanations"]):
    print(f"Explanation {i + 1}: {explanation}")

### Calculator needed

In [ ]:
query = QueryWithContext(query="Sum 2+59", history=[])

answer = await complex_agent.answer(query)
assert answer.content[0].text is not None
answer_as_json = json.loads(answer.content[0].text)

print("\nAnswer: ", answer_as_json["answer"])
print("\nExplanations: ")
for i, explanation in enumerate(answer_as_json["explanations"]):
    print(f"Explanation {i + 1}: {explanation}")

### Text Tool needed

In [ ]:
query = QueryWithContext(
    query="Rewrite the following text with every other word bolded: 'This is an example of the half bold text tool in action.'",
    history=[],
)

answer = await complex_agent.answer(query)
assert answer.content[0].text is not None
answer_as_json = json.loads(answer.content[0].text)

print("\nAnswer: ", answer_as_json["answer"])
print("\nExplanations: ")
for i, explanation in enumerate(answer_as_json["explanations"]):
    print(f"Explanation {i + 1}: {explanation}")

### Both Tools

In [ ]:
query = QueryWithContext(
    query="First sum 10 and 20, then rewrite the result with every other word bolded by writing a short story about the result (1 sentence).",
    history=[],
)

answer = await complex_agent.answer(query)
assert answer.content[0].text is not None
answer_as_json = json.loads(answer.content[0].text)

print("\nAnswer: ", answer_as_json["answer"])
print("\nExplanations: ")
for i, explanation in enumerate(answer_as_json["explanations"]):
    print(f"Explanation {i + 1}: {explanation}")

---

## Conversation with an Agent

The agent can be used in a multi-turn conversation by passing previous exchanges as `history`. Only the final answer (not the intermediate tool calls) is kept in the history. This keeps context lean while preserving the conversational thread.

In [ ]:
# Let's create thr history message from the query
user_initial_message = LLMMessage(
    role=Roles.USER, content=[MessageContent(type="text", text=query.query)]
)
history_messages = [user_initial_message, answer]

# And create a new query
new_query = QueryWithContext(
    query="Make a joke based on past conversation.", history=history_messages
)

final_answer = await complex_agent.answer(new_query)
assert final_answer.content[0].text is not None
final_answer_as_json = json.loads(final_answer.content[0].text)

print("\nAnswer: ", final_answer_as_json["answer"])
print("\nExplanations: ")
for i, explanation in enumerate(final_answer_as_json["explanations"]):
    print(f"Explanation {i + 1}: {explanation}")

------------------------